
# DSI Deconvolution (DSID) vs DSI

An alternative method to DSI is the method proposed by
:footcite:p:`CanalesRodriguez2010` which is called DSI with Deconvolution. This
algorithm is using Lucy-Richardson deconvolution in the diffusion propagator
with the goal to create sharper ODFs with higher angular resolution.

In this example we will show with simulated data how this method's ODF
performs against standard DSI ODF and a ground truth multi tensor ODF.


In [ ]:
import numpy as np

from dipy.core.gradients import gradient_table
from dipy.data import get_fnames, get_sphere
from dipy.reconst.dsi import DiffusionSpectrumDeconvModel, DiffusionSpectrumModel
from dipy.reconst.shm import sf_to_sh
from dipy.sims.voxel import multi_tensor, multi_tensor_odf
from dipy.viz import window
from dipy.viz.skyline import create_shm_visualization

For the simulation we will use a standard DSI acquisition scheme with 514
gradient directions and 1 S0.



In [ ]:
btable = np.loadtxt(get_fnames(name="dsi515btable"))

gtab = gradient_table(btable[:, 0], bvecs=btable[:, 1:])

Let's create a multi tensor with 2 fiber directions at 60 degrees.



In [ ]:
evals = np.array([[0.0015, 0.0003, 0.0003], [0.0015, 0.0003, 0.0003]])

directions = [(-30, 0), (30, 0)]

fractions = [50, 50]

signal, _ = multi_tensor(
    gtab, evals, S0=100, angles=directions, fractions=fractions, snr=None
)

sphere = get_sphere(name="repulsion724").subdivide(n=1)

odf_gt = multi_tensor_odf(
    sphere.vertices, evals, angles=directions, fractions=fractions
)

Perform the reconstructions with standard DSI and DSI with deconvolution.



In [ ]:
dsi_model = DiffusionSpectrumModel(gtab)

dsi_odf = dsi_model.fit(signal).odf(sphere)

dsid_model = DiffusionSpectrumDeconvModel(gtab)

dsid_odf = dsid_model.fit(signal).odf(sphere)

Finally, we can visualize the ground truth ODF, together with the DSI and DSI
with deconvolution ODFs and observe that with the deconvolved method it is
easier to resolve the correct fiber directions because the ODF is sharper.



In [ ]:
# Colors encode orientation. SH coefficients approximate the sampled ODFs for
# display; each ODF is normalized independently without changing the original.
# Viewing from +Y exposes the X/Z crossing plane without rotating the data.

odfs = np.vstack((odf_gt, dsi_odf, dsid_odf))[:, None, None]
odfs /= odfs.max(axis=-1, keepdims=True)
coeffs = sf_to_sh(
    odfs,
    sphere,
    sh_order_max=16,
    basis_type="descoteaux07",
    full_basis=False,
    legacy=False,
)
odf_visualization = create_shm_visualization(
    (coeffs, None, "DSI comparison", "descoteaux07"),
    0,
    l_max=16,
    scale=0.5,
)

show_manager = window.ShowManager(title="dsid", size=(200, 200))
screen = show_manager.screens[0]
screen.scene.add(odf_visualization.actor)
window.set_camera_from_axis(screen, (0, 1, 0))
screen.camera.zoom = 2 / 3
show_manager.start()

.. rst-class:: centered small fst-italic fw-semibold

Ground truth ODF (left), DSI ODF (middle), DSI with Deconvolution ODF (right).


## References

.. footbibliography::




.. include:: ../../links_names.inc


